#### Imports

In [3]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import SparkSession
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import count



#### Starte Session und lade Daten

In [4]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)
df_ratings = spark.read.csv("ml-25m/ratings.csv",header=True, inferSchema=True)

# Entfernt Filme mit weniger als 50 Bewertungen
popular_movies = (
    df_ratings
    .groupBy("movieId")
    .agg(count("*").alias("n_ratings"))
    .filter("n_ratings >= 50")
)

df_ratings = df_ratings.join(
    popular_movies.select("movieId"),
    on="movieId"
)
df_ratings.show()

+-------+------+------+----------+
|movieId|userId|rating| timestamp|
+-------+------+------+----------+
|   1088|     1|   4.0|1147868495|
|   1580|     2|   4.5|1141417059|
|   3175|     2|   3.5|1141417288|
|  44022|     3|   4.0|1439473713|
| 175197|     3|   3.5|1566089493|
|   1580|     4|   4.5|1573938142|
|   3175|     4|   4.0|1573943913|
|   1580|     8|   5.0| 890510873|
|   1645|     8|   4.0| 890510977|
|   1088|     9|   5.0| 859384754|
|    471|    12|   4.0|1167582388|
|   1088|    12|   4.0|1167582465|
|   1580|    12|   3.5|1167582669|
|   3794|    12|   2.0|1137230098|
|   8638|    12|   4.0|1149092439|
|  33722|    12|   3.0|1159805048|
|   1580|    13|   3.5|1238029138|
|   2142|    13|   4.5|1278602136|
|   2366|    13|   3.5|1279481249|
|   3175|    13|   4.0|1278443226|
+-------+------+------+----------+
only showing top 20 rows


#### Baseline Model Training

In [6]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
)

train_df, val_df = df_ratings.randomSplit([0.8, 0.2], seed=42)
model = als.fit(train_df)
predictions = model.transform(val_df)
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")

model.save("results/models/als_25m_base")

RMSE: 0.8037211408260264


#### Hyperparameter Tunning

In [7]:
paramGrid = ParamGridBuilder().addGrid(als.rank, [10, 20, 50]).addGrid(als.regParam, [0.01, 0.1, 1.0]).build()
cv = CrossValidator(
    estimator=als, 
    estimatorParamMaps=paramGrid,
    evaluator=evaluator, 
    numFolds=3
)
cvModel = cv.fit(train_df)

bestModel = cvModel.bestModel

print("Best rank:", bestModel.rank)
print("Best regParam:", bestModel._java_obj.parent().getRegParam())
bestModel.save("results/models/als_25m_final")


Best rank: 20
Best regParam: 0.1


In [ ]:
from pyspark.sql.functions import collect_set, expr, avg, lit, array_intersect, size


relevant = (
    val_df
    .filter("rating >= 4.0")
    .groupBy("userId")
    .agg(collect_set("movieId").alias("relevant_movies"))
)

k = 10

recommendations = (
    bestModel
    .recommendForAllUsers(k)
)


recommendations = recommendations.withColumn(
    "recommended_movies",
    expr("transform(recommendations, x -> x.movieId)")
)

eval_df = recommendations.join(
    relevant,
    on="userId",
    how="inner"
)

eval_df = eval_df.withColumn(
    "hits",
    size(
        array_intersect(
            "recommended_movies",
            "relevant_movies"
        )
    )
)

# Precision

precision = (
    eval_df
    .withColumn("precision", eval_df.hits / lit(k))
    .agg(avg("precision"))
    .first()[0]
)

print("Precision@10 =", precision)

# Recall

eval_df = eval_df.withColumn(
    "n_relevant",
    size("relevant_movies")
)


recall = (
    eval_df
    .withColumn(
        "recall",
        eval_df.hits / eval_df.n_relevant
    )
    .agg(avg("recall"))
    .first()[0]
)

print("Recall@10 =", recall)
